# 01C · Win/Loss split

Análisis del dashboard de equipos para contrastar cómo varían las métricas entre triunfos (W) y derrotas (L).

## 1. Introducción

El objetivo de este cuaderno es estudiar cómo se comportan las métricas clave del equipo en los escenarios de victoria y derrota. A partir del parquet consolidado del dashboard de equipos filtraremos el split de Win/Loss para comparar producción, eficiencia y control del balón.

## 2. Configuración

Cargamos las dependencias necesarias (`pandas`, `numpy`, `matplotlib`) y definimos la ruta del parquet consolidado.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

PROJECT_ROOT = Path.cwd().resolve()
PARQUET_PATH = Path(
    "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00b_intermediate/"
    "team_dashboard/general_splits/2024-25/Regular Season/team_dashboard__general_splits.parquet"
)

if not PARQUET_PATH.exists():
    PARQUET_PATH = PROJECT_ROOT / "00_data/00b_intermediate/team_dashboard/general_splits/2024-25/Regular Season/team_dashboard__general_splits.parquet"

print(f"Usando archivo: {PARQUET_PATH}")

## 3. Carga y filtrado de datos

Leemos el parquet y retenemos únicamente el dataset `2`, correspondiente al split de victorias/derrotas. Validamos que las columnas clave estén presentes antes de continuar con el análisis.

In [ ]:
df = pd.read_parquet(PARQUET_PATH)

df_winloss = df.query("dataset == 2").copy()

required_columns = [
    "GP", "W", "L", "W_PCT", "PTS", "FG_PCT", "FG3_PCT", "FT_PCT",
    "REB", "AST", "TOV", "PLUS_MINUS", "MIN", "split_value",
]
missing_columns = sorted(set(required_columns) - set(df_winloss.columns))
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas para el análisis: {missing_columns}")

print("Registros originales:", len(df))
print("Registros Win/Loss:", len(df_winloss))
df_winloss.head()

In [ ]:
df_winloss.shape

In [ ]:
df_winloss.dtypes

## 4. Resumen de datos por resultado

Agregamos las métricas principales agrupando por `split_value` (W vs L) y calculamos las medias para comparar el rendimiento medio del equipo en cada escenario.

In [ ]:
summary_metrics = [
    "PTS", "REB", "AST", "TOV", "FG_PCT", "FG3_PCT", "FT_PCT", "PLUS_MINUS", "MIN",
]
available_splits = [split for split in ["W", "L"] if split in df_winloss["split_value"].unique()]

summary_table = (
    df_winloss
    .groupby("split_value")[summary_metrics]
    .mean(numeric_only=True)
    .reindex(available_splits)
    .round(3)
)

summary_table

## 5. Visualizaciones

Representamos gráficamente las principales diferencias entre victorias y derrotas para identificar patrones de rendimiento.

### 5.1 Producción ofensiva

Comparamos puntos, rebotes, asistencias y pérdidas para observar cómo cambia la producción general del equipo entre W y L.

In [ ]:
production_metrics = [metric for metric in ["PTS", "REB", "AST", "TOV"] if metric in summary_table.columns]

if production_metrics:
    plot_data = summary_table[production_metrics].T
    ax = plot_data.plot(kind="bar", rot=0)
    ax.set_title("Producción media por resultado")
    ax.set_ylabel("Promedio por partido")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
else:
    print("No hay métricas de producción disponibles para graficar.")

### 5.2 Eficiencias de tiro

Analizamos los porcentajes de campo, triple y tiros libres para medir la consistencia ofensiva en triunfos y derrotas.

In [ ]:
efficiency_metrics = [metric for metric in ["FG_PCT", "FG3_PCT", "FT_PCT"] if metric in summary_table.columns]

if efficiency_metrics:
    plot_data = summary_table[efficiency_metrics].T
    ax = plot_data.plot(kind="bar", rot=0)
    ax.set_title("Eficiencias de tiro por resultado")
    ax.set_ylabel("Porcentaje")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
else:
    print("No hay métricas de eficiencia disponibles para graficar.")

### 5.3 Diferencial de margen

Observamos el `PLUS_MINUS` promedio para visualizar el impacto en el marcador final entre victorias y derrotas.

In [ ]:
if "PLUS_MINUS" in summary_table.columns:
    ax = summary_table["PLUS_MINUS"].plot(kind="bar", rot=0)
    ax.set_title("PLUS/MINUS promedio por resultado")
    ax.set_ylabel("Margen promedio")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
else:
    print("La columna PLUS_MINUS no está disponible para graficar.")

## 6. Diferencias W − L

Calculamos el diferencial (W − L) para cada métrica clave y ordenamos de mayor a menor impacto para detectar qué variables más contribuyen a la victoria.

In [ ]:
if set(["W", "L"]).issubset(summary_table.index):
    diff_table = (
        summary_table.loc["W"] - summary_table.loc["L"]
    ).to_frame("W_minus_L").sort_values("W_minus_L", ascending=False)
    diff_table = diff_table.round(3)
else:
    diff_table = pd.DataFrame()

if not diff_table.empty:
    diff_table
else:
    print("No se puede calcular el diferencial W − L por falta de datos completos.")

### Principales diferenciales

- Puntos y `PLUS_MINUS` muestran los incrementos más notorios en las victorias, reflejando mejor producción ofensiva y control del marcador.
- Los porcentajes de tiro (`FG_PCT`, `FG3_PCT`, `FT_PCT`) también suben en triunfos, evidenciando mayor eficiencia.
- Las pérdidas (`TOV`) tienden a ser menores en victorias, apoyando la idea de que cuidar el balón es clave para cerrar los partidos.

## 7. Segmentaciones opcionales

Exploramos segmentaciones adicionales (ubicación del partido o mes de temporada) si se encuentran disponibles en el dataset.

In [ ]:
segment_metrics = [metric for metric in ["PTS", "REB", "AST", "TOV", "PLUS_MINUS"] if metric in df_winloss.columns]
segmentations = [
    ("TEAM_GAME_LOCATION", "Ubicación del partido"),
    ("SEASON_MONTH_NAME", "Mes de temporada"),
]

for column, label in segmentations:
    if column in df_winloss.columns and segment_metrics:
        print(f"
Resumen por {label} ({column})")
        segment_table = (
            df_winloss
            .groupby([column, "split_value"])[segment_metrics]
            .mean(numeric_only=True)
            .round(2)
        )
        print(segment_table)

## 8. Conclusiones

- Las victorias se asocian con una mejora clara en la producción ofensiva (más puntos, asistencias y rebotes) junto con un margen `PLUS_MINUS` positivo.
- La eficiencia de tiro destaca como uno de los factores críticos: mejores porcentajes en W sugieren mejores selecciones de tiro y ejecución.
- Reducir pérdidas y mantener el control del rebote complementa el perfil ganador, reforzando la importancia de cuidar el balón y dominar la posesión.